### 시장 규모 및 성장

In [62]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag, NavigableString

url = "https://dream.kotra.or.kr/kotranews/cms/news/actionKotraBoardDetail.do?SITE_NO=3&MENU_ID=200&CONTENTS_NO=1&bbsGbn=403&bbsSn=403&pNttSn=231078"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text):
    text = text.replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = text.replace("\u200b", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_date(soup):
    # txtInfo 전체에서 날짜 패턴만 추출
    info = soup.select_one("div.board_area div.txtInfo")
    if info:
        text = clean_text(info.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    # fallback: li.date가 있으면 거기서 추출
    node = soup.select_one("li.date")
    if node:
        text = clean_text(node.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    return ""


def parse_table(table_tag):
    rows = []

    for tr in table_tag.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        row = [clean_text(c.get_text(" ", strip=True)) for c in cells]

        if any(row):
            rows.append(row)

    if not rows:
        return None

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    if len(rows) >= 2:
        return pd.DataFrame(rows[1:], columns=rows[0])
    else:
        return pd.DataFrame(rows)


def extract_content(soup):
    body = soup.select_one("#pdfArea .view_txt .viewDataWrap")
    if body is None:
        return []

    content = []

    for child in body.children:

        # 텍스트 노드
        if isinstance(child, NavigableString):
            text = clean_text(str(child))
            if text:
                content.append(text)
            continue

        if not isinstance(child, Tag):
            continue

        # table 바로 있는 경우
        if child.name == "table":
            df = parse_table(child)
            if df is not None:
                content.append(df)
            continue

        # div 안 table
        inner_table = child.find("table")
        if inner_table:
            clone = BeautifulSoup(str(child), "html.parser")

            for t in clone.find_all("table"):
                t.decompose()

            text = clean_text(clone.get_text(" ", strip=True))
            if text:
                content.append(text)

            df = parse_table(inner_table)
            if df is not None:
                content.append(df)

            continue

        # 일반 텍스트
        text = clean_text(child.get_text(" ", strip=True))
        if text:
            content.append(text)

    return content


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title_node = soup.select_one("div.board_area div.txtL strong")
title = clean_text(title_node.get_text(" ", strip=True)) if title_node else ""

date = extract_date(soup)

content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("=" * 100)

# ------------------------
# content 출력
# ------------------------
for block in content:
    print("\n")

    if isinstance(block, str):
        print(block)
    else:
        print("[TABLE]\n")
        print(block)

TITLE: 2025년 미국 화장품 산업정보
DATE: 2025-07-01


1. 산업 특성


1.1 정책 및 규제


미국에서 화장품을 제조하거나 수입· 유통·판매하기 위해서는 ‘연방 식품∙의약품∙화장품법(Food Drug & Cosmetics Act)’과 ‘공정 포장 및 라벨링법(FPLA)’을 철저히 준수해야 한다. 여기에 더해 2022년 말 제정된 ‘화장품 규제 현대화법(이하 MOCRA, Modernization of Cosmetics Regulation Act of 2022)’이 2023년부터 단계적으로 시행됨에 따라 미국 화장품 규제는 수십 년 만에 가장 큰 변화를 맞이했다.


MOCRA 는 FDA에 더 많은 감독 권한을 부여하고, 제조업체 및 유통업체에 새로운 의무를 부과하고 있다. 주요 내용으로는 화장품 제조시설 등록 의무화, 제품 및 성분 목록 제출, 심각한 부작용 보고 의무, 우수제조관리기준(cGMP) 적용, 라벨에 연락처 및 특정 알레르기 유발 성분 표시 등이 포함된다. 이로 인해 해외 기업이 미국 시장에 화장품을 수출하려면 제품 라벨링과 안전성, 제조공정 등에 대한 규정 준수를 보다 철저히 준비해야 한다.


1.2 주요 기업 현황


2025 년 현재 미국 화장품 제조 시장은 로레알USA(62억 달러), 유니레버(49억 달러), P&G(44억 달러), 에스티로더(40억 달러) 등 글로벌 대기업들이 시장의 상당 부분을 점유하고 있다. 이들 기업은 대규모 R&D 투자와 글로벌 유통망, 강력한 브랜드 포트폴리오를 통해 산업을 주도하고 있으며, 소비자 세분화 전략과 채널 다변화 전략을 동시에 활용하고 있다.


특히 로레알은 NYX, 랑콤, 메이블린 등 다양한 브랜드를 보유하고 있어 가격대별·연령대별 소비자 니즈에 유연하게 대응하고 있다. 로레알은 프랑스가 본사지만 글로벌 로컬 생산 전략을 통해 판매하는 주요국마다 현지 생산 비중을 높이고 있다. 미국에서도 아칸소주와 뉴저지주 등에 생산 공장을 운영하며 미국 시장 맞춤형 제품을 생산하기 위해

In [11]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag, NavigableString

url = "https://www.grandviewresearch.com/industry-analysis/us-skin-care-products-market-report"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def is_noise_text(text: str) -> bool:
    if not text:
        return True

    low = text.lower().strip()

    bad_exact = {
        "table of contents",
        "segmentation",
        "methodology",
        "download free sample",
        "download sample report",
        "buy now",
        "request sample",
        "report summary",
        "interactive charts",
    }

    bad_contains = [
        "this site uses cookies",
        "download free sample",
        "download sample report",
        "to learn more about this report",
        "request sample",
        "buy now",
        "recent visited",
        "my reports",
    ]

    if low in bad_exact:
        return True

    if any(b in low for b in bad_contains):
        return True

    if len(text) <= 1:
        return True

    return False


def extract_title(soup: BeautifulSoup) -> str:
    candidates = [
        "meta[property='og:title']",
        "h1",
        "title",
    ]

    for selector in candidates:
        node = soup.select_one(selector)
        if node:
            if node.name == "meta":
                content = node.get("content", "")
                if content:
                    return clean_text(content)
            else:
                txt = clean_text(node.get_text(" ", strip=True))
                if txt:
                    return txt
    return ""


def extract_published_date(soup: BeautifulSoup) -> str:
    # 보통 Grand View는 정확한 published date가 잘 안 잡히는 편
    text = soup.get_text(" ", strip=True)

    patterns = [
        r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\s+\d{1,2},\s+\d{4}\b",
        r"\b\d{4}-\d{2}-\d{2}\b",
    ]

    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            return m.group(0)

    return ""


def parse_table(table_tag: Tag):
    rows = []

    for tr in table_tag.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        row = [clean_text(cell.get_text(" ", strip=True)) for cell in cells]
        if any(row):
            rows.append(row)

    return rows


def rows_to_dataframe(rows):
    if not rows:
        return pd.DataFrame()

    max_cols = max(len(r) for r in rows)
    padded_rows = [r + [""] * (max_cols - len(r)) for r in rows]

    if len(padded_rows) >= 2:
        return pd.DataFrame(padded_rows[1:], columns=padded_rows[0])

    return pd.DataFrame(padded_rows)


def extract_content_in_order(soup: BeautifulSoup):
    """
    report_summary.full 내부를 위에서 아래로 순서대로 읽어서
    text / table 을 content 흐름대로 추출
    """
    body = soup.select_one("div.report_summary.full")

    if not body:
        return []

    content = []

    for child in body.children:
        if isinstance(child, NavigableString):
            text = clean_text(str(child))
            if text and not is_noise_text(text):
                content.append(text)
            continue

        if not isinstance(child, Tag):
            continue

        # table 바로 있는 경우
        if child.name == "table":
            rows = parse_table(child)
            if rows:
                content.append(rows_to_dataframe(rows))
            continue

        # div 안에 table이 있는 경우
        inner_table = child.find("table")
        if inner_table:
            clone = BeautifulSoup(str(child), "html.parser")
            for t in clone.find_all("table"):
                t.decompose()

            text = clean_text(clone.get_text(" ", strip=True))
            if text and not is_noise_text(text):
                content.append(text)

            rows = parse_table(inner_table)
            if rows:
                content.append(rows_to_dataframe(rows))
            continue

        # 일반 텍스트
        text = clean_text(child.get_text(" ", strip=True))
        if text and not is_noise_text(text):
            content.append(text)

    return content


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title = extract_title(soup)
published_date = extract_published_date(soup)
content = extract_content_in_order(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", published_date)
print("CONTENT BLOCK COUNT:", len(content))
print("=" * 100)

for idx, block in enumerate(content, start=1):
    print("\n" + "=" * 100)
    print(f"BLOCK {idx}")

    if isinstance(block, str):
        print(block)
    else:
        print("[TABLE]\n")
        print(block.head())

TITLE: U.S. Skin Care Products Market Size & Share | Report, 2030
DATE: 
CONTENT BLOCK COUNT: 3

BLOCK 1
U.S. Skin Care Products Market Trends The U.S. skin care products market size was estimated at USD 22.90 billion in 2023 and is expected to grow at a compound annual growth rate (CAGR) of 4.2% from 2024 to 2030. The market growth is attributed to the large number of individuals spend a considerable amount on skin care and personal grooming in the country.Growing awareness regarding one’s physical appearance is expected to spur the demand for various skin care products such as face creams, face washes, and cleansers over the forecast period. U.S. skin care products market accounted for the share of 16.10% of the global skin care products market in 2023. Consumers in the region prefer prominent e-commerce players such as Amazon and Walmart to buy most of their daily use items, including skin care and beauty products, as they offer various benefits like the convenience of free home del

In [12]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag, NavigableString

url = "https://www.grandviewresearch.com/industry-analysis/beauty-personal-care-products-market"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def is_noise_text(text: str) -> bool:
    if not text:
        return True

    low = text.lower().strip()

    bad_exact = {
        "table of contents",
        "segmentation",
        "methodology",
        "download free sample",
        "download sample report",
        "buy now",
        "request sample",
        "report summary",
        "interactive charts",
    }

    bad_contains = [
        "this site uses cookies",
        "download free sample",
        "download sample report",
        "to learn more about this report",
        "request sample",
        "buy now",
        "recent visited",
        "my reports",
    ]

    if low in bad_exact:
        return True

    if any(b in low for b in bad_contains):
        return True

    if len(text) <= 1:
        return True

    return False


def extract_title(soup: BeautifulSoup) -> str:
    candidates = [
        "meta[property='og:title']",
        "h1",
        "title",
    ]

    for selector in candidates:
        node = soup.select_one(selector)
        if node:
            if node.name == "meta":
                content = node.get("content", "")
                if content:
                    return clean_text(content)
            else:
                txt = clean_text(node.get_text(" ", strip=True))
                if txt:
                    return txt
    return ""


def extract_published_date(soup: BeautifulSoup) -> str:
    # 보통 Grand View는 정확한 published date가 잘 안 잡히는 편
    text = soup.get_text(" ", strip=True)

    patterns = [
        r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\s+\d{1,2},\s+\d{4}\b",
        r"\b\d{4}-\d{2}-\d{2}\b",
    ]

    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            return m.group(0)

    return ""


def parse_table(table_tag: Tag):
    rows = []

    for tr in table_tag.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        row = [clean_text(cell.get_text(" ", strip=True)) for cell in cells]
        if any(row):
            rows.append(row)

    return rows


def rows_to_dataframe(rows):
    if not rows:
        return pd.DataFrame()

    max_cols = max(len(r) for r in rows)
    padded_rows = [r + [""] * (max_cols - len(r)) for r in rows]

    if len(padded_rows) >= 2:
        return pd.DataFrame(padded_rows[1:], columns=padded_rows[0])

    return pd.DataFrame(padded_rows)


def extract_content_in_order(soup: BeautifulSoup):
    """
    report_summary.full 내부를 위에서 아래로 순서대로 읽어서
    text / table 을 content 흐름대로 추출
    """
    body = soup.select_one("div.report_summary.full")

    if not body:
        return []

    content = []

    for child in body.children:
        if isinstance(child, NavigableString):
            text = clean_text(str(child))
            if text and not is_noise_text(text):
                content.append(text)
            continue

        if not isinstance(child, Tag):
            continue

        # table 바로 있는 경우
        if child.name == "table":
            rows = parse_table(child)
            if rows:
                content.append(rows_to_dataframe(rows))
            continue

        # div 안에 table이 있는 경우
        inner_table = child.find("table")
        if inner_table:
            clone = BeautifulSoup(str(child), "html.parser")
            for t in clone.find_all("table"):
                t.decompose()

            text = clean_text(clone.get_text(" ", strip=True))
            if text and not is_noise_text(text):
                content.append(text)

            rows = parse_table(inner_table)
            if rows:
                content.append(rows_to_dataframe(rows))
            continue

        # 일반 텍스트
        text = clean_text(child.get_text(" ", strip=True))
        if text and not is_noise_text(text):
            content.append(text)

    return content


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title = extract_title(soup)
published_date = extract_published_date(soup)
content = extract_content_in_order(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", published_date)
print("CONTENT BLOCK COUNT:", len(content))
print("=" * 100)

for idx, block in enumerate(content, start=1):
    print("\n" + "=" * 100)
    print(f"BLOCK {idx}")

    if isinstance(block, str):
        print(block)
    else:
        print("[TABLE]\n")
        print(block.head())

TITLE: Beauty and Personal Care Products Market Report, 2030
DATE: 
CONTENT BLOCK COUNT: 3

BLOCK 1
Beauty And Personal Care Products Market Summary The global beauty and personal care products market size was estimated at USD 557.24 billion in 2023 and is projected to reach USD 937.13 billion by 2030, growing at a CAGR of 7.7% from 2024 to 2030. One of the primary factors driving market expansion is the rising consciousness of consumers about their appearance. Key Market Trends & Insights The Asia Pacific beauty and personal care products market dominated the global market with a revenue share of 39.3% in 2023. The Australian market is projected to grow at a CAGR of around 8.5% from 2024 to 2030. By type, the conventional segment beauty and personal care market accounted for the largest revenue share of 84.6% in 2023. By product, the skincare segment accounted for a revenue share of 33.7% of the beauty and personal care products industry in 2023. Market Size & Forecast 2023 Market Siz

### K-뷰티 브랜드 시장 위치

In [32]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag, NavigableString
from urllib.parse import urljoin

url = "https://www.grandviewresearch.com/industry-analysis/k-beauty-products-market-report"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def is_noise_text(text: str) -> bool:
    if not text:
        return True

    low = text.lower().strip()

    bad_exact = {
        "table of contents",
        "segmentation",
        "methodology",
        "download free sample",
        "download sample report",
        "buy now",
        "request sample",
        "report summary",
        "interactive charts",
    }

    bad_contains = [
        "this site uses cookies",
        "download free sample",
        "download sample report",
        "to learn more about this report",
        "request sample",
        "buy now",
        "recent visited",
        "my reports",
    ]

    if low in bad_exact:
        return True

    if any(b in low for b in bad_contains):
        return True

    if len(text) <= 1:
        return True

    return False


def extract_title(soup: BeautifulSoup) -> str:
    candidates = [
        "meta[property='og:title']",
        "h1",
        "title",
    ]

    for selector in candidates:
        node = soup.select_one(selector)
        if node:
            if node.name == "meta":
                content = node.get("content", "")
                if content:
                    return clean_text(content)
            else:
                txt = clean_text(node.get_text(" ", strip=True))
                if txt:
                    return txt
    return ""


def extract_published_date(soup: BeautifulSoup) -> str:
    # 보통 Grand View는 정확한 published date가 잘 안 잡히는 편
    text = soup.get_text(" ", strip=True)

    patterns = [
        r"\b(?:Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Sept|Oct|Nov|Dec)[a-z]*\s+\d{1,2},\s+\d{4}\b",
        r"\b\d{4}-\d{2}-\d{2}\b",
    ]

    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            return m.group(0)

    return ""


def parse_table(table_tag: Tag):
    rows = []

    for tr in table_tag.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        row = [clean_text(cell.get_text(" ", strip=True)) for cell in cells]
        if any(row):
            rows.append(row)

    return rows


def extract_images(soup: BeautifulSoup, base_url: str):
    images = []

    for img in soup.select("img"):
        src = img.get("src")
        if src:
            images.append(urljoin(base_url, src))

    return images

def rows_to_dataframe(rows):
    if not rows:
        return pd.DataFrame()

    max_cols = max(len(r) for r in rows)
    padded_rows = [r + [""] * (max_cols - len(r)) for r in rows]

    if len(padded_rows) >= 2:
        return pd.DataFrame(padded_rows[1:], columns=padded_rows[0])

    return pd.DataFrame(padded_rows)


def extract_content_in_order(soup: BeautifulSoup):
    """
    report_summary.full 내부를 위에서 아래로 순서대로 읽어서
    text / table 을 content 흐름대로 추출
    """
    body = soup.select_one("div.report_summary.full")

    if not body:
        return []

    content = []

    for child in body.children:
        if isinstance(child, NavigableString):
            text = clean_text(str(child))
            if text and not is_noise_text(text):
                content.append(text)
            continue

        if not isinstance(child, Tag):
            continue

        # table 바로 있는 경우
        if child.name == "table":
            rows = parse_table(child)
            if rows:
                content.append(rows_to_dataframe(rows))
            continue

        # div 안에 table이 있는 경우
        inner_table = child.find("table")
        if inner_table:
            clone = BeautifulSoup(str(child), "html.parser")
            for t in clone.find_all("table"):
                t.decompose()

            text = clean_text(clone.get_text(" ", strip=True))
            if text and not is_noise_text(text):
                content.append(text)

            rows = parse_table(inner_table)
            if rows:
                content.append(rows_to_dataframe(rows))
            continue

        # 일반 텍스트
        text = clean_text(child.get_text(" ", strip=True))
        if text and not is_noise_text(text):
            content.append(text)

    return content


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title = extract_title(soup)
published_date = extract_published_date(soup)
content = extract_content_in_order(soup)
images = extract_images(soup, url)

print("=" * 100)
print("TITLE:", title)
print("DATE:", published_date)
print("CONTENT BLOCK COUNT:", len(content))
print("=" * 100)

for idx, block in enumerate(content, start=1):
    print("\n" + "=" * 100)
    print(f"BLOCK {idx}")

    if isinstance(block, str):
        print(block)
    else:
        print("[TABLE]\n")
        print(block.head())



TITLE: K-Beauty Products Market Size, Share | Industry Report 2033
DATE: 
CONTENT BLOCK COUNT: 3

BLOCK 1
K-beauty Products Market Summary The global K-beauty products market size was estimated at USD 118.28 billion in 2025 and is expected to reach USD 252.41 billion by 2033 , growing at a CAGR of 10.0% from 2026 to 2033 . The global rise of Korean pop culture, including K-pop and K-dramas, has played a significant role in expanding K-beauty worldwide, as consumers increasingly adopt skin care and makeup trends inspired by Korean celebrities and influencers. Key Market Trends & Insights By product, skin care led the K-beauty products market and accounted for a share of 56.78% in 2025. By end user, women led the K-beauty products market and accounted for a share of 61.81% in 2025. By distribution channel, specialty stores led the K-beauty products market and accounted for a share of 44.96% in 2025. By region, North America led the K-beauty products market and accounted for a share of 34

In [16]:
import re
import requests
from bs4 import BeautifulSoup

url = "https://www.euromonitor.com/article/k-beautys-global-footprint-performance-presence-and-strategic-strengths?utm_source=chatgpt.com"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_title(soup: BeautifulSoup) -> str:
    node = soup.select_one("h1.e-page-heading")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    return ""


def extract_date(soup: BeautifulSoup) -> str:
    node = soup.select_one("span.e-published-date")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    return ""


def extract_content(soup: BeautifulSoup) -> str:
    article = soup.select_one("article.e-editable-area")
    if not article:
        return ""

    for tag in article.select("script, style"):
        tag.decompose()

    text = article.get_text(separator="\n")
    text = clean_text(text)

    return text


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title = extract_title(soup)
date = extract_date(soup)
content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("CONTENT LENGTH:", len(content))
print("=" * 100)
print(content)

TITLE: K-Beauty’s Global Footprint: Performance, Presence, and Strategic Strengths
DATE: 12/4/2025
CONTENT LENGTH: 4024
K-beauty’s global trajectory has entered a new chapter. Once dominated by China, the export landscape has shifted dramatically, with the US now commanding over half of total sales. K-beauty 1.0, spanning roughly 2011-2018, highlighted Korean skin care routines and innovative items like BB cream. In contrast, K-beauty 2.0, emerging from 2020, emphasises advanced technology, stronger global branding, and mid-range products that deliver high quality at accessible prices.
This article is based on Euromonitor E-Commerce data through to Q2 2025. Download the full-version white paper 
here
, with the most updated Q3 2025 E-Commerce data.
The market reversal from China to the US
The US has overtaken China as the top export destination for K-beauty, accounting for 55% of total K-beauty’s overseas online sales in the first half-year of 2025, up from 20% in 2022, according to Eu

In [19]:
import re
import requests
from bs4 import BeautifulSoup, Tag

url = "https://www.euromonitor.com/newsroom/press-releases/december-2025/k-beauty-2.0-surges-online-beauty-sales-expected-to-surpass-2024-total-sales"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_title(soup: BeautifulSoup) -> str:
    node = soup.select_one("h1.e-press-release__title")
    return clean_text(node.get_text(" ", strip=True)) if node else ""


def extract_date(soup: BeautifulSoup) -> str:
    node = soup.select_one("span.e-date")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    return ""


def extract_content(soup: BeautifulSoup) -> str:
    body = soup.select_one("article.e-press-release .row .col-sm-7.col-md-9")
    if not body:
        return ""

    blocks = []

    # 직계 자식 기준 순회
    for child in body.find_all(recursive=False):
        if not isinstance(child, Tag):
            continue

        # Contact Us 섹션 시작 태그 만나면 중단
        if child.name == "p" and "e-small-paragraph" in (child.get("class") or []):
            break

        text = clean_text(child.get_text(" ", strip=True))
        if text:
            blocks.append(text)

    return "\n".join(blocks).strip()


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title = extract_title(soup)
date = extract_date(soup)
content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("CONTENT LENGTH:", len(content))
print("=" * 100)
print(content)

TITLE: K-beauty 2.0 surges: Online beauty sales expected to surpass 2024 total sales
DATE: 10 December 2025
CONTENT LENGTH: 3885
Online sales reach 86% of the 2024 total in just nine months of this year The US takes 51% of global K-beauty sales, Europe more than triples to 11% 87 brands generate USD 1M+, five exceed USD100M annually In Euromonitor International’s report ‘Glass Skin & Global Wins: The Rise of K-Beauty ’ , K-beauty 2.0 features advanced technology, stronger brand positioning and proven quality at fair prices, unlike K-beauty’s first wave focusing on niche skin care rituals. This evolution aligns with today's value-conscious consumers seeking better performance without premium pricing. K-beauty online sales in 15 countries outside of South Korea are projected to exceed the 2024 total. Yang Hu , Asia Pacific health and beauty insight manager at Euromonitor International , said: “K-beauty 2.0 represents a pivotal shift in the global beauty market. Distributors have a clear 

### 성분, 제형, 기능 트렌드 제시

In [73]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag, NavigableString

url = "https://dream.kotra.or.kr/kotranews/cms/news/actionKotraBoardDetail.do?SITE_NO=3&MENU_ID=180&CONTENTS_NO=1&bbsGbn=243&bbsSn=243&pNttSn=214738"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text):
    text = text.replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = text.replace("\u200b", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_date(soup):
    # txtInfo 전체에서 날짜 패턴만 추출
    info = soup.select_one("div.board_area div.txtInfo")
    if info:
        text = clean_text(info.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    # fallback: li.date가 있으면 거기서 추출
    node = soup.select_one("li.date")
    if node:
        text = clean_text(node.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    return ""


def parse_table(table_tag):
    rows = []

    for tr in table_tag.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        row = [clean_text(c.get_text(" ", strip=True)) for c in cells]

        if any(row):
            rows.append(row)

    if not rows:
        return None

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    if len(rows) >= 2:
        return pd.DataFrame(rows[1:], columns=rows[0])
    else:
        return pd.DataFrame(rows)


def extract_content(soup):
    body = soup.select_one("#pdfArea .view_txt .viewDataWrap")
    if body is None:
        return []

    content = []

    for child in body.children:

        # 텍스트 노드
        if isinstance(child, NavigableString):
            text = clean_text(str(child))
            if text:
                content.append(text)
            continue

        if not isinstance(child, Tag):
            continue

        # table 바로 있는 경우
        if child.name == "table":
            df = parse_table(child)
            if df is not None:
                content.append(df)
            continue

        # div 안 table
        inner_table = child.find("table")
        if inner_table:
            clone = BeautifulSoup(str(child), "html.parser")

            for t in clone.find_all("table"):
                t.decompose()

            text = clean_text(clone.get_text(" ", strip=True))
            if text:
                content.append(text)

            df = parse_table(inner_table)
            if df is not None:
                content.append(df)

            continue

        # 일반 텍스트
        text = clean_text(child.get_text(" ", strip=True))
        if text:
            content.append(text)

    return content


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title_node = soup.select_one("div.board_area div.txtL strong")
title = clean_text(title_node.get_text(" ", strip=True)) if title_node else ""

date = extract_date(soup)

content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("=" * 100)

# ------------------------
# content 출력
# ------------------------
for block in content:
    print("\n")

    if isinstance(block, str):
        print(block)
    else:
        print("[TABLE]\n")
        print(block)

TITLE: '웰니스' 키워드로 보는 미국 스킨케어 시장 동향
DATE: 2024-05-08


바야흐로 ‘웰니스(Wellness)’의 전성시대다. 최근 몇 년간 코로나19 팬데믹으로 인한 깊은 피로감은 많은 미국인들이 신체적, 정서적으로 건강하고 행복한 삶을 추구하는 ‘웰니스’를 삶의 중요한 가치로 인식 하게 되는 계기가 됐 다. 이러한 ‘ 웰니스 ’ 중심의 생활 방식은 미국인들의 일상적인 습관에서부터 소비 패턴에 이르기까지 깊은 영향을 미치고 있다 . 웰니스가 단순한 유행을 넘어 삶의 필수적인 부분으로 자리 잡게 되면서 , 뷰티· 스킨케어 업계 또한 변화하는 소비자 수요에 맞춰 새로운 제품군을 선보이며 ‘ 웰니스 ’ 시장 공략에 나서고 있다 .


하나의 화장품으로 여러 가지 효과를, ‘멀티태스킹’ 화장품 전성시대


최근 미국 뷰티시장에서는 ‘스킨스트리밍(Skin Streaming)’이 새로운 트렌드로 각광받고 있다. 스킨 스트리밍이란 세안부터 보습에 이르기까지 많게는 십여단계에 이르는 과도한 스킨케어 루틴을 줄이고, 소비자 개개인의 피부 상태에 맞는 최소한의 제품 사용을 통해 효과를 극대화하는 스킨케어 방식으로, 피부로 가는 과도한 영양 공급을 제한해 피부 자극을 줄이는 동시에 스킨케어에 소모되는 비용 및 시간을 절감할 수 있어 바쁜 현대인들에게 이상적인 스킨케어 방식으로 각광받고 있다.


스킨스트리밍 트렌드에 맞춰, 업계는 보다 효율적이고 간소화 된 제품 개발에 주력하고 있으며, 많은 브랜드들은 보습 , 자외선 차단 등 여러 가지 기능을 포함한 ‘ 멀티태스킹 (Multitasking)’ 제품을 시장에 선보이고 있다 . 이는 소비자들의 스킨케어 루틴을 단순화하는 동시에 피부 관리에 필수적인 기능을 모두 포함해 특히 바쁜 일상 속의 현대인들에게 큰 인기를 얻고 있다 .


<멀티테스킹 스킨케어 제품>


[TABLE]

  사진                                         주요 기능 및 비고 가격(U$)
0     - 제품명:

In [75]:
import re
import requests
from bs4 import BeautifulSoup, Tag

url = "https://www.euromonitor.com/article/beauty-consumer-trends-key-insights-from-the-voice-of-the-consumer-survey"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
}


def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\r", "\n")
    text = text.replace("\u200b", "")
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_title(soup: BeautifulSoup) -> str:
    node = soup.select_one("h1.e-page-heading")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    node = soup.select_one("h1")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    return ""


def extract_date(soup: BeautifulSoup) -> str:
    node = soup.select_one("span.e-published-date")
    if node:
        return clean_text(node.get_text(" ", strip=True))

    text = soup.get_text("\n", strip=True)
    m = re.search(r"\b\d{1,2}/\d{1,2}/\d{4}\b", text)
    if m:
        return m.group(0)

    return ""


def extract_content(soup: BeautifulSoup) -> str:
    article = soup.select_one("article.e-editable-area")
    if not article:
        return ""

    clone_soup = BeautifulSoup(str(article), "html.parser")
    article_clone = clone_soup.select_one("article.e-editable-area")
    if not article_clone:
        return ""

    # 차트 링크/스크립트 제거
    for tag in article_clone.select("script, style, a[href*='contentassets']"):
        tag.decompose()

    blocks = []

    # article 내부 직계 자식만 순서대로 읽기
    for child in article_clone.find_all(recursive=False):
        if not isinstance(child, Tag):
            continue

        # 섹션 제목
        if child.name == "h2":
            text = clean_text(child.get_text(" ", strip=True))
            if text:
                blocks.append(text)
            continue

        # 일반 문단
        if child.name == "p":
            text = clean_text(child.get_text(" ", strip=True))
            if not text:
                continue
            if text in {"Bio", "Source:"}:
                continue
            blocks.append(text)
            continue

    # 중복 제거
    cleaned = []
    prev = None
    for block in blocks:
        if block != prev:
            cleaned.append(block)
        prev = block

    return "\n".join(cleaned).strip()


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title = extract_title(soup)
date = extract_date(soup)
content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("CONTENT LENGTH:", len(content))
print("=" * 100)
print(content[:10000])

TITLE: Beauty Consumer Trends: Key Insights from the Voice of the Consumer Survey
DATE: 11/11/2025
CONTENT LENGTH: 4556
In 2025, beauty consumers globally have been more intentional and informed than ever. Over the past five years, consumer focus has shifted from routine-driven habits and product replacement towards strategic, personalised care. Euromonitor International’s Voice of the Consumer: Beauty Survey, fielded in May 2025, reflects that beauty consumers are making conscious choices, prefer clean ingredients, and seek beauty products backed by scientific evidence. Consumers are taking beauty into their own hands by approaching self-care strategically, and integrating beauty with overall wellbeing.
Beauty as a long-term investment in health and longevity
Consumers in 2025 view beauty as an investment in their overall health and longevity. 75% of people agree that having a consistent beauty routine contributes to their overall wellbeing and confidence. This mindset means that cons

In [63]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag, NavigableString

url = "https://dream.kotra.or.kr/dream/cms/news/actionKotraBoardDetail.do?SITE_NO=2&MENU_ID=3530&bbsSn=254&pNttSn=234165&CONTENTS_NO=1"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text):
    text = text.replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = text.replace("\u200b", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_date(soup):
    # txtInfo 전체에서 날짜 패턴만 추출
    info = soup.select_one("div.board_area div.txtInfo")
    if info:
        text = clean_text(info.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    # fallback: li.date가 있으면 거기서 추출
    node = soup.select_one("li.date")
    if node:
        text = clean_text(node.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    return ""


def parse_table(table_tag):
    rows = []

    for tr in table_tag.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        row = [clean_text(c.get_text(" ", strip=True)) for c in cells]

        if any(row):
            rows.append(row)

    if not rows:
        return None

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    if len(rows) >= 2:
        return pd.DataFrame(rows[1:], columns=rows[0])
    else:
        return pd.DataFrame(rows)


def extract_content(soup):
    body = soup.select_one("#pdfArea .view_txt .viewDataWrap")
    if body is None:
        return []

    content = []

    for child in body.children:

        # 텍스트 노드
        if isinstance(child, NavigableString):
            text = clean_text(str(child))
            if text:
                content.append(text)
            continue

        if not isinstance(child, Tag):
            continue

        # table 바로 있는 경우
        if child.name == "table":
            df = parse_table(child)
            if df is not None:
                content.append(df)
            continue

        # div 안 table
        inner_table = child.find("table")
        if inner_table:
            clone = BeautifulSoup(str(child), "html.parser")

            for t in clone.find_all("table"):
                t.decompose()

            text = clean_text(clone.get_text(" ", strip=True))
            if text:
                content.append(text)

            df = parse_table(inner_table)
            if df is not None:
                content.append(df)

            continue

        # 일반 텍스트
        text = clean_text(child.get_text(" ", strip=True))
        if text:
            content.append(text)

    return content


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title_node = soup.select_one("div.board_area div.txtL strong")
title = clean_text(title_node.get_text(" ", strip=True)) if title_node else ""

date = extract_date(soup)

content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("=" * 100)

# ------------------------
# content 출력
# ------------------------
for block in content:
    print("\n")

    if isinstance(block, str):
        print(block)
    else:
        print("[TABLE]\n")
        print(block)

TITLE: 미국 기초화장품 시장동향
DATE: 2025-09-15


상품명 및 HS Code


<상품명 및 HS Code>


[TABLE]

     상품명  HS Code
0  기초화장품  3304.99


시 장 동향


미국 기초화장품 시장은 2024년 연 매출 249억 달러를 기록하며 전년 대비 0.6% 성장했다. 같은 해 전 세계 화장품 시장에서 북미는 34%의 점유율을 차지하며 글로벌 시장을 주도했으며, 이 중 기초화장품이 가장 큰 비중을 차지한다. 이에 따라 미국 기초화장품 시장의 주요 트렌드를 살펴보고자 한다.


1. 혁신 기술


다양한 피부 타입 및 톤에 맞춤 제품에 대한 개발이 강화되고 있다. 특히, AI의 발전에 따라서 기초화장품 시장 역시 맞춤형 서비스가 이용가능해졌고, 소비자들은 개인적인 니즈에 따라 제품을 추천받고, 사용하려는 추세를 보이고 있다. 세계적인 브랜드 로레알은 모디페이스(ModiFace)와 같은 기술기업과 협력하여 AI와 증강현실(AR)기술을 접목하여 소비자들이 개인 맞춤형 스킨케어 추천을 받을 수 있다. 또한, 50세 이상의 성인인구 및 30대 여성들 역시, 노화 징후가 나타나기 전에 미리 안티에이징 제품을 사용하는 예방 중심 트렌드가 확산되면서, 과학적 기술 기반 안티에이징 제품에 대한 지출이 증가하고 있다.


2. 지속가능 및 클린뷰티


특히, 브랜드 구매 결정 요인이 단순한 기능성에서 지속가능성으로 확대되는 추세이다. 친환경 원료, 윤리적 공급망, 재활용 및 생분해성 포장재를 강조하는 브랜드를 선호하는 심리가 강화되고 있다. 이러한 추세에 맞춰서 주요 화장품 제조업체브랜드는 성분을 명확하게 공개하고, 유해 가능성이 있는 화학성분을 제외한 제품을 출시 및 마켓팅하고 있다. 특히, 클린뷰티란, 건강친화적 성분 및 윤리적 제조방식을 결합한 단어로 미국 스킨케어 시장의 주요한 트렌드로 자리잡고 있다. 이에 따라서, 미국 소비자들에게 각광받고있는 주요 성분으로는 히알루로닉에이시드, 나이아신마이드, 스쿠알란, 호

In [20]:
import re
import requests
from bs4 import BeautifulSoup

url = "https://www.personalcareinsights.com/news/spate-reveals-2025-beauty-trends-skinification-and-multifunctional-solutions-top-the-charts.html"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_title(soup: BeautifulSoup) -> str:
    node = soup.select_one("h1.heading")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    return ""


def extract_date(soup: BeautifulSoup) -> str:
    node = soup.select_one("div.author.source-color.article-title-block span.source-color")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    return ""


def extract_author(soup: BeautifulSoup) -> str:
    nodes = soup.select("div.author.source-color.article-title-block span.source-color")
    if len(nodes) >= 2:
        return clean_text(nodes[1].get_text(" ", strip=True))
    return ""


def extract_content(soup: BeautifulSoup) -> str:
    body = soup.select_one("div.news-text")
    if not body:
        return ""

    for tag in body.select("script, style"):
        tag.decompose()

    lines = []
    bad_phrases = [
        "by continuing to browse our site you agree",
        "privacy statement",
        "i agree",
    ]

    for element in body.get_text(separator="\n").split("\n"):
        line = clean_text(element)
        if not line:
            continue

        low = line.lower()
        if any(bad in low for bad in bad_phrases):
            continue

        lines.append(line)

    return "\n".join(lines).strip()


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title = extract_title(soup)
date = extract_date(soup)
author = extract_author(soup)
content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("AUTHOR:", author)
print("CONTENT LENGTH:", len(content))
print("=" * 100)
print(content)

TITLE: Spate reveals 2025 beauty trends: Skinification and multifunctional solutions top the charts
DATE: 16 Dec 2024
AUTHOR: | By Sabine Waldeck
CONTENT LENGTH: 6782
The skinification of body care, texture-driven experiences and fragrance layering are the key beauty trends to look out for in the coming year. Spate releases its beauty trend predictions for 2025 and tells brands to focus on multi-functional products that simplify routines.
The machine intelligence platform expects a growing emphasis on category convergence, with body care increasingly borrowing from skin care to create multi-benefit solutions. “Brands that embrace this shift can tap into expanding markets and redefine how body care is perceived,” says Spate.
The Spate’s
dashboard analyzes
over 20 billion search signals and over 60 million beauty-related TikTok videos worldwide. It also implements artificial intelligence to identify trend clusters.
Skin type tracking
Consumers are reshaping their preferences for cosmetic

### 주요 온 · 오프라인 유통 채널

In [76]:
import re
import requests
from bs4 import BeautifulSoup


url = "https://hamacher.com/the-beauty-industry-is-growing-both-in-store-and-online/"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "en-US,en;q=0.9",
}


def clean_text(text):
    text = text.replace("\r", "\n")
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


res = requests.get(url, headers=headers, timeout=30)
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")


# -------------------------
# TITLE
# -------------------------
title = ""
title_node = soup.select_one("h1.elementor-heading-title")

if title_node:
    title = clean_text(title_node.get_text())


# -------------------------
# CONTENT
# -------------------------
content_blocks = []

article = soup.select_one('div[data-widget_type="theme-post-content.default"]')

if article:

    for p in article.select("p"):

        text = clean_text(p.get_text())

        if not text:
            continue

        content_blocks.append(text)


content = "\n".join(content_blocks)


print("=" * 100)
print("TITLE:", title)
print("CONTENT LENGTH:", len(content))
print("=" * 100)
print(content)

TITLE: The beauty industry is growing — both in store and online
CONTENT LENGTH: 3351
by Megan Moyer, corporate marketing manager, as appeared in Inside Beauty for Drug Store News
There is a positive outlook for the beauty industry and it’s being driven by online shoppers. While TikTok Shops have gained popularity, some surprising categories are having the greatest success.
In a recent CEW (Cosmetic Executive Women) webinar, “Global Trends Report, Part 2,” Nielsen IQ shared that beauty continues to grow globally, up +10% worldwide. In a deeper dive, their data showed that spend per shopper had increased and units purchased per shopper increased, as did the frequency of shopping trips.
Additionally, they showed that online beauty sales were driving growth more so than in-store sales with online sales up +19% compared to in-store sales +2%.
Data from Nielsen IQ indicated that Amazon is the number one online merchant for beauty sales. Pyxis, the consumer analytics division of Bain & Compa

### 주요 경쟁 브랜드

In [26]:
import re
import requests
from bs4 import BeautifulSoup

url = "https://www.grandviewresearch.com/voice-of-consumer/us-skincare-products-category-insights"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_title(soup: BeautifulSoup) -> str:
    node = soup.select_one("div.iner_about_heading h1")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    return ""


def extract_content(soup: BeautifulSoup) -> str:
    body = soup.select_one("div.advanced_report_list.full.press_room")
    if not body:
        return ""

    lines = []

    for child in body.find_all(recursive=False):
        text = clean_text(child.get_text(" ", strip=True))
        if not text:
            continue

        low = text.lower()

        # 하단 CTA/공유 영역 전 제거
        if (
            "how this report will benefit you" in low
            or "talk to a specialist" in low
            or low == "share"
            or low == "e-mail"
            or low == "save"
            or low == "print"
        ):
            break

        lines.append(text)

    return "\n".join(lines).strip()


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title = extract_title(soup)
content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("CONTENT LENGTH:", len(content))
print("=" * 100)
print(content)

TITLE: U.S. Skincare Products Consumer Insights - Usage, Attitudes, And Brand Performance Report
CONTENT LENGTH: 3980
Introduction
Skin care consumers are evolving globally, and the U.S. is at the forefront of this evolution. Thanks to the free flow of information all around, consumers today are more knowledgeable, aware, well-read, and hence less driven by mere promotions and visuals. Brands need to keep a tap on the changing pulse of the consumers to stay relevant. Skin care is one of the most commonly adopted categories in the U.S. market, with 8 out of 10 adults using skin care products at least once a day .
Methodology
Insights covered in this report are drawn from GVR’s ‘Voice of Consumer Survey-2024’ and its periodic updates. The latest survey represents 75,000+ consumer interviews conducted across 20 countries for 100+ product categories . These insights are designed to help Skin Care brands in their strategic decision-making process. Our VoC reports provide insights around cat

In [36]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

url = "https://www.mjsmedicals.com/what-is-the-most-popular-skincare-brand-in-the-usa/"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text: str) -> str:
    text = text.replace("\r", "\n")
    text = re.sub(r"\u00a0", " ", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_title(soup: BeautifulSoup) -> str:
    node = soup.select_one("h1.entry-title.title")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    return ""


def extract_date(soup: BeautifulSoup) -> str:
    node = soup.select_one("li.meta-date")
    if node:
        text = clean_text(node.get_text(" ", strip=True))
        return text.replace("On ", "").strip()
    return ""


def parse_table(table):
    rows = []

    for tr in table.select("tr"):
        cells = tr.select("th, td")
        row = [clean_text(cell.get_text(" ", strip=True)) for cell in cells]
        if any(row):
            rows.append(row)

    return rows


def rows_to_df(rows):
    if not rows:
        return pd.DataFrame()

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    if len(rows) >= 2:
        return pd.DataFrame(rows[1:], columns=rows[0])
    return pd.DataFrame(rows)


def extract_content(soup: BeautifulSoup) -> str:
    body = soup.select_one("div.elementor-widget-theme-post-content div.elementor-widget-container")
    if not body:
        return ""

    # 표 영역은 본문 텍스트에서 제거하고 텍스트만 추출
    body_clone = BeautifulSoup(str(body), "html.parser")

    for tag in body_clone.select("script, style, table, .hyc-common-markdown__table-wrapper"):
        tag.decompose()

    lines = []
    bad_phrases = [
        "table of contents",
        "posted by",
    ]

    for element in body_clone.get_text(separator="\n").split("\n"):
        line = clean_text(element)
        if not line:
            continue

        low = line.lower()
        if any(bad in low for bad in bad_phrases):
            continue

        lines.append(line)

    return "\n".join(lines).strip()


def extract_tables(soup: BeautifulSoup):
    body = soup.select_one("div.elementor-widget-theme-post-content div.elementor-widget-container")
    if not body:
        return []

    tables = []

    for idx, table in enumerate(body.select("div.hyc-common-markdown__table-wrapper table, table"), start=1):
        rows = parse_table(table)
        if rows:
            tables.append({
                "table_index": idx,
                "rows": rows,
                "dataframe": rows_to_df(rows)
            })

    return tables


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title = extract_title(soup)
date = extract_date(soup)
content = extract_content(soup)
tables = extract_tables(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("CONTENT LENGTH:", len(content))
print("TABLE COUNT:", len(tables))
print("=" * 100)

print("\n[CONTENT]\n")
print(content)

for table_obj in tables:
    print("\n" + "=" * 100)
    print(f"TABLE {table_obj['table_index']} RAW ROWS")
    for row in table_obj["rows"][:5]:
        print(row)

    print(f"\nTABLE {table_obj['table_index']} DATAFRAME PREVIEW\n")
    print(table_obj["dataframe"].head())

TITLE: What is the most popular skincare brand in the USA?
DATE: 2025-08-13
CONTENT LENGTH: 10705
TABLE COUNT: 2

[CONTENT]

CeraVe is currently the top skincare brand in the U.S., with 28% market share in drugstores (Nielsen 2024). Its dermatologist-developed formulas, like the $15 Hyaluronic Acid Serum, drive 35% annual growth, fueled by TikTok trends and partnerships with 50,000+ skincare professionals nationwide.
Toggle
​​Top Picks by Sales​​
​​Customer Favorites Ranked​​
​​Best for Sensitive Skin​​
​​Drugstore vs. Luxury Brands​
​​What Experts Recommend​​
​
​Top Picks by Sales​
​
When it comes to skincare in the U.S., sales numbers don’t lie. In 2023, the American skincare market was valued at
18.6 billion
, with the top 5 brands accounting for
34% of sales
.
CeraVe
led with
1.2 billion in sales
last year alone, thanks to its dermatologist-recommended formulas and affordable pricing (
10-25 per product
). Close behind is
Neutrogena
, with
950 million in annual sales
, driven by it

In [49]:
import re
import requests
from bs4 import BeautifulSoup, Tag

url = "https://www.fortunebusinessinsights.com/skin-care-market-102544"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\r", "\n")
    text = text.replace("\u200b", "")
    text = text.replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_title(soup: BeautifulSoup) -> str:
    node = soup.select_one("h1.mabt10.mt10.critical-h1")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    node = soup.select_one("h1")
    if node:
        return clean_text(node.get_text(" ", strip=True))
    return ""


def extract_date(soup: BeautifulSoup) -> str:
    text = soup.get_text("\n", strip=True)

    patterns = [
        r"Last Updated:\s*([A-Z][a-z]+ \d{1,2}, \d{4})",
        r"Last Updated\s*([A-Z][a-z]+ \d{1,2}, \d{4})",
        r"\b([A-Z][a-z]+ \d{1,2}, \d{4})\b",
    ]

    for pattern in patterns:
        m = re.search(pattern, text, flags=re.IGNORECASE)
        if m:
            return clean_text(m.group(1))
    return ""


def extract_content(soup: BeautifulSoup) -> str:
    # 1순위: summary 탭
    body = soup.select_one("#summary")

    # fallback
    if not body:
        body = soup.select_one("div.tab-content")

    if not body:
        return ""

    body_clone = BeautifulSoup(str(body), "html.parser")

    # 표/폼 제거
    for tag in body_clone.select("table, form, input, select, textarea, button, script, style"):
        tag.decompose()

    # 본문 후보 태그들만 순서대로 수집
    candidates = body_clone.select("h2, h3, p, li, center")

    bad_phrases = [
        "download free sample",
        "buy now",
        "submit",
        "listen to audio version",
        "get 20% free customization",
        "growth advisory services",
        "go top",
        "we use cookies",
        "accept",
        "security code",
        "full name",
        "business email",
        "phone no",
        "customize this report",
        "expand regional and country coverage",
        "how can we help you uncover new opportunities",
        "check now",
    ]

    blocks = []

    for el in candidates:
        if not isinstance(el, Tag):
            continue

        text = clean_text(el.get_text(" ", strip=True))
        if not text:
            continue

        low = text.lower()
        if any(bad in low for bad in bad_phrases):
            continue

        # chartTitle 류 제거
        classes = el.get("class", [])
        if "chartTitle" in classes:
            continue

        if el.name == "li":
            text = f"- {text}"

        blocks.append(text)

    # 중복 제거
    cleaned = []
    prev = None
    for b in blocks:
        if b != prev:
            cleaned.append(b)
        prev = b

    return "\n".join(cleaned).strip()


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title = extract_title(soup)
date = extract_date(soup)
content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("CONTENT LENGTH:", len(content))
print("=" * 100)
print(content[:8000])

# 디버그용
print("\n[DEBUG]")
print("HAS #summary:", bool(soup.select_one("#summary")))
print("HAS .tab-content:", bool(soup.select_one("div.tab-content")))

TITLE: Skincare Market Size, Share & Industry Analysis, By Product (Creams, Lotions, Powders, Sprays, and Others), Packaging Type (Tube, Bottle, Jar, and Others), Gender (Men and Women), Distribution Channel (Cosmetic Stores, Supermarkets/Hypermarkets, Online Channels, and Others), and Regional Forecast, 2026 - 2034
DATE: February 23, 2026
CONTENT LENGTH: 20182
Skincare Industry Analysis
The global skincare market size was valued at USD 122.11 billion in 2025. The market is projected to grow from USD 129.11 billion in 2026 to USD 227.13 billion by 2034, exhibiting a CAGR of 7.32% during the forecast period. Asia Pacific dominated the skin care market with a market share of 51.46% in 2025. Moreover, the skincare market size in the U.S. is projected to grow significantly, reaching an estimated value of USD 30.42 billion by 2032, driven by rising demand for organic and natural products to surge product demand.
Self-caring products include creams, lotions, and powders, which improve the qu

In [64]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag, NavigableString

url = "https://dream.kotra.or.kr/kotranews/cms/news/actionKotraBoardDetail.do?SITE_NO=3&MENU_ID=180&CONTENTS_NO=1&bbsGbn=243&bbsSn=243&pNttSn=228244"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text):
    text = text.replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = text.replace("\u200b", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_date(soup):
    # txtInfo 전체에서 날짜 패턴만 추출
    info = soup.select_one("div.board_area div.txtInfo")
    if info:
        text = clean_text(info.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    # fallback: li.date가 있으면 거기서 추출
    node = soup.select_one("li.date")
    if node:
        text = clean_text(node.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    return ""


def parse_table(table_tag):
    rows = []

    for tr in table_tag.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        row = [clean_text(c.get_text(" ", strip=True)) for c in cells]

        if any(row):
            rows.append(row)

    if not rows:
        return None

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    if len(rows) >= 2:
        return pd.DataFrame(rows[1:], columns=rows[0])
    else:
        return pd.DataFrame(rows)


def extract_content(soup):
    body = soup.select_one("#pdfArea .view_txt .viewDataWrap")
    if body is None:
        return []

    content = []

    for child in body.children:

        # 텍스트 노드
        if isinstance(child, NavigableString):
            text = clean_text(str(child))
            if text:
                content.append(text)
            continue

        if not isinstance(child, Tag):
            continue

        # table 바로 있는 경우
        if child.name == "table":
            df = parse_table(child)
            if df is not None:
                content.append(df)
            continue

        # div 안 table
        inner_table = child.find("table")
        if inner_table:
            clone = BeautifulSoup(str(child), "html.parser")

            for t in clone.find_all("table"):
                t.decompose()

            text = clean_text(clone.get_text(" ", strip=True))
            if text:
                content.append(text)

            df = parse_table(inner_table)
            if df is not None:
                content.append(df)

            continue

        # 일반 텍스트
        text = clean_text(child.get_text(" ", strip=True))
        if text:
            content.append(text)

    return content


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title_node = soup.select_one("div.board_area div.txtL strong")
title = clean_text(title_node.get_text(" ", strip=True)) if title_node else ""

date = extract_date(soup)

content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("=" * 100)

# ------------------------
# content 출력
# ------------------------
for block in content:
    print("\n")

    if isinstance(block, str):
        print(block)
    else:
        print("[TABLE]\n")
        print(block)

TITLE: 미국에서 제2의 전성기 맞은 K-뷰티
DATE: 2025-04-24


지난해 K-뷰티는 미국 시장에서 괄목할 만한 성장을 이뤄냈다. 2024년 한국은 미국 화장품 수입 시장에서 오랜 기간 1위를 지켜온 프랑스를 제치고 선두에 올랐다. 미국 시장에서 K-뷰티 붐을 일으킨 지 10여 년 만이다. 한국 화장품 대미 수출은 2416만 달러를 기록한 2006년 이후 지속적인 증가세를 이어왔다. 마스크팩과 BB크림, 달팽이 크림 같은 특정 제품이 미국 시장에서 크게 주목을 받으며 미 소비자들에게 존재감을 알린 K-뷰티는 새롭고 혁신적인 제품을 찾는 MZ 세대를 공략하며 입지를 넓혀 나갔다.


한국 , 미국으로의 화장품 최대 수출국으로 도약


지난해 한국의 화장품 대미 수출 규모는 전년 대비 54.3% 증가한 17억100만 달러를 기록했다. 미국 화장품 수입시장 점유율은 전년보다 5.9%포인트 상승한 22.4%로 나타났다. 한국 화장품이 미 수입시장에서 1위에 오른 것은 이번이 처음이다. 오랜 라이벌인 프랑스는 수출액이 전년 대비 9.6% 증가하는 데 그쳐 12억6300만 달러를 기록, 한국에 1위 자리를 내줬다.


<미국의 화장품 수입 현황>


(단위: US$ 백만, %)


[TABLE]

      순위     국가명    수입액     비중 ‘23~’24년 증감                        
0   2022    2023   2024   2022        2023  2024                  
1      1    대한민국    824  1,103       1,701  13.5  16.5  22.4  54.3
2      2     프랑스  1,052  1,152       1,263  17.2  17.3  16.6   9.7
3      3     캐나다    907  1,033       1,022  14.8  15.5  13.5  -1.0
4      4    이탈리아    706    825         

In [61]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag, NavigableString

url = "https://dream.kotra.or.kr/dream/cms/news/actionKotraBoardDetail.do?SITE_NO=2&MENU_ID=3550&CONTENTS_NO=1&bbsGbn=243&bbsSn=243&pNttSn=223837"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text):
    text = text.replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = text.replace("\u200b", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_date(soup):
    # txtInfo 전체에서 날짜 패턴만 추출
    info = soup.select_one("div.board_area div.txtInfo")
    if info:
        text = clean_text(info.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    # fallback: li.date가 있으면 거기서 추출
    node = soup.select_one("li.date")
    if node:
        text = clean_text(node.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    return ""


def parse_table(table_tag):
    rows = []

    for tr in table_tag.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        row = [clean_text(c.get_text(" ", strip=True)) for c in cells]

        if any(row):
            rows.append(row)

    if not rows:
        return None

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    if len(rows) >= 2:
        return pd.DataFrame(rows[1:], columns=rows[0])
    else:
        return pd.DataFrame(rows)


def extract_content(soup):
    body = soup.select_one("#pdfArea .view_txt .viewDataWrap")
    if body is None:
        return []

    content = []

    for child in body.children:

        # 텍스트 노드
        if isinstance(child, NavigableString):
            text = clean_text(str(child))
            if text:
                content.append(text)
            continue

        if not isinstance(child, Tag):
            continue

        # table 바로 있는 경우
        if child.name == "table":
            df = parse_table(child)
            if df is not None:
                content.append(df)
            continue

        # div 안 table
        inner_table = child.find("table")
        if inner_table:
            clone = BeautifulSoup(str(child), "html.parser")

            for t in clone.find_all("table"):
                t.decompose()

            text = clean_text(clone.get_text(" ", strip=True))
            if text:
                content.append(text)

            df = parse_table(inner_table)
            if df is not None:
                content.append(df)

            continue

        # 일반 텍스트
        text = clean_text(child.get_text(" ", strip=True))
        if text:
            content.append(text)

    return content


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title_node = soup.select_one("div.board_area div.txtL strong")
title = clean_text(title_node.get_text(" ", strip=True)) if title_node else ""

date = extract_date(soup)

content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("=" * 100)

# ------------------------
# content 출력
# ------------------------
for block in content:
    print("\n")

    if isinstance(block, str):
        print(block)
    else:
        print("[TABLE]\n")
        print(block)

TITLE: 바이어에게 듣는 미국 화장품 시장 트렌드
DATE: 2024-12-23


미국에서 K-Beauty의 인기가 지속적으로 높아지고 있다. Landing International에서 최근 발표된 보고서에 의하면, 2015~2016년경 미국에서의 첫 번째 K-Beauty 붐이 시작됐고, 2024년에는 더 큰 인기로 두 번째 붐을 맞이하고 있다. 올해 1분기 미국의 한국 화장품 수입은 4억 7 710만 달러로, 프랑스를 제치고 점유율 1위를 차지했다 . 한국 제품은 안전하고 효과가 좋다는 인식이 정착되면서 K-뷰티 시장의 성장은 지속될 것으로 전망된다.


글로벌 시장조사 기업 Straits Research는 2023년 전 세계 한국 화장품 시장 규모를 125억4000만 달러로 평가했으며, 연평균 8.4%의 증가율로 성장해 2032년에는 259억80 00만 달러에 달할 것으로 예측했다.


<전 세계 K-Beauty 제품 시장 전망>


(단위: US$ 십억, %)


[자료: Strait Research]


미국 현지 화장품 업체와의 인터뷰


미국 내 한국 화장품의 인기가 늘어나는 가운데, KOTRA 애틀랜타 무역관에서는 현지에서 한국제품의 온라인 판매 및 B2B 사업을 운영하는 E 사의 대표와의 인터뷰를 통해서 미국 뷰티 시장의 트렌드 및 전망에 대한 정보를 들어봤다.


Q1. 먼저, 귀하의 회사에 관한 간단한 소개를 부탁드립니다.


A1. 우리 회사는 귀여운 아이템과 진정한 한국 화장품을 정성껏 선별해 제공하는 선물 가게입니다. 저희는 박람회와 축제에 참가하는 팝업 숍으로 사업을 시작했으며, 현재는 온라인 판매와 B2B 파트너십으로 사업 방향을 확장하고 있습니다. 우리 회사의 목표는 고객에게 매력적이고 고품질인 제품을 제공하며, 직접 소매와 도매 시장 모두에서 저희의 존재감을 계속해서 키워 나가는 것입니다.


Q2. 최근 미국 뷰티 시장의 주요 트렌드는 무엇인가요?


A2. 지속 가능성, 제품 품질에 대한 집중, 그리고 쇼핑 방식의 변화가

In [71]:
import re
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

url = "https://www.linkedin.com/posts/jontilley_the-new-q3-2025-beauty-data-from-pattern-activity-7404926138483019776-uECT"


def clean_text(text: str) -> str:
    if not text:
        return ""
    text = text.replace("\r", "\n").replace("\u200b", "").replace("\xa0", " ")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


async def main():
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page(viewport={"width": 1440, "height": 2600})

        try:
            await page.goto(url, wait_until="load", timeout=30000)
        except PlaywrightTimeoutError:
            print("goto timeout -> continuing")

        try:
            await page.wait_for_selector("h1.section-title.mb-2", timeout=15000)
        except PlaywrightTimeoutError:
            print("title selector timeout")

        await page.wait_for_timeout(3000)

        title = ""
        content = ""

        title_loc = page.locator("h1.section-title.mb-2")
        if await title_loc.count() > 0:
            title = clean_text(await title_loc.first.inner_text())

        for selector in [
            "p[data-test-id='main-feed-activity-card__commentary']",
            "p.attributed-text-segment-list__content",
        ]:
            loc = page.locator(selector)
            if await loc.count() > 0:
                content = clean_text(await loc.first.inner_text())
                if content:
                    break

        print("=" * 100)
        print("TITLE:", title)
        print("CONTENT LENGTH:", len(content))
        print("=" * 100)
        print(content)

        await browser.close()


await main()

TITLE: Amazon Beauty Data: Top Brands by Category Q3 2025
CONTENT LENGTH: 2137
The new Q3 2025 beauty data from Pattern shows how different categories behave on Amazon, and the mix of players is wider than it looks at first glance.
𝗠𝗮𝗸𝗲𝘂𝗽 𝗶𝘀 𝘁𝗵𝗲 𝗺𝗼𝘀𝘁 𝗰𝗼𝗻𝗰𝗲𝗻𝘁𝗿𝗮𝘁𝗲𝗱 𝗰𝗮𝘁𝗲𝗴𝗼𝗿𝘆.
Maybelline leads with 6%, and brands like e.l.f., Laura Geller and NYX follow close behind.
It is mostly mass brands with high recognition and strong review volume.
Grande Cosmetics is the only independent brand in the group.
𝗦𝗸𝗶𝗻𝗰𝗮𝗿𝗲 𝗶𝘀 𝗺𝗼𝗿𝗲 𝗼𝗽𝗲𝗻.
The top 10 brands hold 19% of sales, and the list spans derm brands like CeraVe and La Roche-Posay, mass staples like Neutrogena and Dove, and rising Korean names like COSRX and Medicube.
Skincare rewards experimentation and routine-building, so smaller brands can gain traction when they show clear results.
𝗙𝗿𝗮𝗴𝗿𝗮𝗻𝗰𝗲 𝗹𝗼𝗼𝗸𝘀 𝗰𝗹𝗼𝘀𝗲𝗿 𝘁𝗼 𝗺𝗮𝗸𝗲𝘂𝗽 𝗶𝗻 𝘁𝗲𝗿𝗺𝘀 𝗼𝗳 𝗰𝗼𝗻𝗰𝗲𝗻𝘁𝗿𝗮𝘁𝗶𝗼𝗻.
Lataffa leads at 6.3%, followed by Versace, Sol de Janeiro and Armaf.
Lataffa’s rise lines up with fast-grow

In [72]:
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup, Tag, NavigableString

url = "https://dream.kotra.or.kr/kotranews/cms/news/actionKotraBoardDetail.do?SITE_NO=3&MENU_ID=180&CONTENTS_NO=1&pNttSn=236751"

headers = {
    "User-Agent": "Mozilla/5.0",
    "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
}


def clean_text(text):
    text = text.replace("\r", "\n")
    text = text.replace("\u00a0", " ")
    text = text.replace("\u200b", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


def extract_date(soup):
    # txtInfo 전체에서 날짜 패턴만 추출
    info = soup.select_one("div.board_area div.txtInfo")
    if info:
        text = clean_text(info.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    # fallback: li.date가 있으면 거기서 추출
    node = soup.select_one("li.date")
    if node:
        text = clean_text(node.get_text(" ", strip=True))
        m = re.search(r"\b20\d{2}-\d{2}-\d{2}\b", text)
        if m:
            return m.group(0)

    return ""


def parse_table(table_tag):
    rows = []

    for tr in table_tag.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        row = [clean_text(c.get_text(" ", strip=True)) for c in cells]

        if any(row):
            rows.append(row)

    if not rows:
        return None

    max_cols = max(len(r) for r in rows)
    rows = [r + [""] * (max_cols - len(r)) for r in rows]

    if len(rows) >= 2:
        return pd.DataFrame(rows[1:], columns=rows[0])
    else:
        return pd.DataFrame(rows)


def extract_content(soup):
    body = soup.select_one("#pdfArea .view_txt .viewDataWrap")
    if body is None:
        return []

    content = []

    for child in body.children:

        # 텍스트 노드
        if isinstance(child, NavigableString):
            text = clean_text(str(child))
            if text:
                content.append(text)
            continue

        if not isinstance(child, Tag):
            continue

        # table 바로 있는 경우
        if child.name == "table":
            df = parse_table(child)
            if df is not None:
                content.append(df)
            continue

        # div 안 table
        inner_table = child.find("table")
        if inner_table:
            clone = BeautifulSoup(str(child), "html.parser")

            for t in clone.find_all("table"):
                t.decompose()

            text = clean_text(clone.get_text(" ", strip=True))
            if text:
                content.append(text)

            df = parse_table(inner_table)
            if df is not None:
                content.append(df)

            continue

        # 일반 텍스트
        text = clean_text(child.get_text(" ", strip=True))
        if text:
            content.append(text)

    return content


# =========================
# 실행
# =========================
res = requests.get(url, headers=headers, timeout=30)
res.raise_for_status()
res.encoding = res.apparent_encoding

soup = BeautifulSoup(res.text, "html.parser")

title_node = soup.select_one("div.board_area div.txtL strong")
title = clean_text(title_node.get_text(" ", strip=True)) if title_node else ""

date = extract_date(soup)

content = extract_content(soup)

print("=" * 100)
print("TITLE:", title)
print("DATE:", date)
print("=" * 100)

# ------------------------
# content 출력
# ------------------------
for block in content:
    print("\n")

    if isinstance(block, str):
        print(block)
    else:
        print("[TABLE]\n")
        print(block)

TITLE: 2026 미국 뷰티 시장 트렌드, K-뷰티 3.0과 스키니멀리즘의 확산
DATE: 2025-11-25


참신함에서 기술 기반 스킨케어 솔루션으로 … K-뷰티 3.0 시대의 도래


아마존을 중심으로 미국 시장에서 빠르게 성장해 온 K-뷰티는 2026년 더욱 영향력을 확대할 것으로 전망된다. K-뷰티가 글로벌 뷰티 트렌드를 선도한다는 인식이 확산되고 있으며, 혁신 기술을 기반으로 한 제품력은 높은 소비자 로열티와 재구매로 이어지고 있다는 분석이다.


2020년 이후 미국에서 론칭되는 스킨케어 제품을 분석한 결과, 미국산 제품은 감소한 반면 한국산 제품은 증가한 것으로 나타났다. 민텔(Mintel)에 따르면 2024년 7월~2025년 6월 미국에서 출시된 미국산 스킨케어 제품은 1484개로, 2020년 7월~2021년 6월 대비 16% 감소했다. 같은 기간 한국산 제품은 496개에서 598개로 20% 증가했다. 민텔은 미국 브랜드들이 해외 제조 의존도를 높이고 있으며, 2021년 이후 혁신 활동이 눈에 띄게 줄었다고 지적한다. 브랜드들이 기존 제품 리뉴얼이나 한정판 출시로 시장 존재감을 유지하려 한 점도 한계로 꼽힌다. 이러한 환경 속에서 시트마스크 등 참신한 제품으로 시장 기반을 마련한 K-뷰티가 기술 중심의 스킨케어 솔루션으로 진화하는 ‘K-뷰티 3.0 시대’가 도래하고 있다는 평가다.


K-뷰티가 미국 시장에서 경쟁력을 확보할 수 있었던 배경에는 다양한 피부색을 고려한 제품 개발과 기술 기반의 합리적 스킨케어 접근 방식이 있다. 복잡한 단계 없이 효율적으로 피부를 케어할 수 있다는 점도 큰 호응을 얻고 있다. 업계 관계자들은 2026년 미국 시장에서 리주란(PN), PDRN 등 한국에서 미용 시술에 활용되는 성분 또는 유사 성분이 더욱 주목받을 것으로 전망했다.


<미국 아마존에서 판매 중인 메디큐브의 PDRN 핑크 펩타이즈 세럼>


[자료: Amazon]


스키니멀리즘 과 하이브리드 메이크업


복잡한 스킨케어 루틴을 간소화하고 최소

In [82]:
import json
import re
import pandas as pd

# 1. 노트북 파일 로드
with open('c.ipynb', 'r', encoding='utf-8') as f:
    nb_data = json.load(f)

def clean_text_refined(text):
    if not text: return ""
    # 불필요한 태그 및 디버그 문구 제거
    text = re.sub(r"BLOCK \d+|\[TABLE\]|\[CONTENT\]", "", text)
    # 파이썬 리스트 형태의 데이터(['...']) 및 기술 문구 제거
    if text.strip().startswith("['") or "RAW ROWS" in text or "DATAFRAME PREVIEW" in text:
        return ""
    # 줄바꿈 및 공백 정리
    text = re.sub(r"\s+", " ", text)
    return text.strip()

final_rows = []

for cell in nb_data.get('cells', []):
    if cell.get('cell_type') != 'code': continue
    
    for out in cell.get('outputs', []):
        if 'text' not in out: continue
        full_text = "".join(out['text'])
        
        if "TITLE:" not in full_text: continue
        
        # [제목] - 너무 긴 경우 핵심만 추출
        title = re.search(r"TITLE:\s*(.*)", full_text).group(1).strip()
        if len(title) > 80:
            title = re.split(r",| - ", title)[0]
            
        # [날짜] - 실제 날짜 형식만 필터링
        date_match = re.search(r"(202[4-6]-[0-9]{2}-[0-9]{2}|[0-9]{1,2} [A-Za-z]+ 202[4-6]|[A-Z][a-z]+ \d{1,2}, 202[4-6]|[0-9/]{8,10})", full_text)
        date = date_match.group(1) if date_match else ""
        
        # [본문] - 의미 있는 긴 텍스트 블록 찾기
        parts = re.split(r"={10,}|BLOCK \d+|\[CONTENT\]", full_text)
        candidate_contents = [clean_text_refined(p) for p in parts if len(clean_text_refined(p)) > 150]
        
        content = candidate_contents[0] if candidate_contents else ""
        
        if title:
            final_rows.append({"Title": title, "Date": date, "Content": content})

# 중복 제거 및 저장
df_final = pd.DataFrame(final_rows).drop_duplicates(subset=['Title'], keep='last')
df_final.to_csv('final_beauty_trends_fixed.csv', index=False, encoding='utf-8-sig')

print(f"✅ 정돈된 데이터 {len(df_final)}건이 'final_beauty_trends_fixed.csv'로 저장되었습니다.")

✅ 정돈된 데이터 18건이 'final_beauty_trends_fixed.csv'로 저장되었습니다.
